In [4]:
# 1. Download directly from the official academic servers (gives a nice progress bar)
!wget -q --show-progress https://homes.cs.washington.edu/~ranjay/visualgenome/data/dataset/scene_graphs.json.zip -O /kaggle/working/scene_graphs.json.zip

# 2. Unzip the file silently into the working directory
!unzip -q /kaggle/working/scene_graphs.json.zip -d /kaggle/working/

# 3. Delete the zip file so it doesn't eat up your Kaggle disk space
!rm /kaggle/working/scene_graphs.json.zip

print("Download and extraction complete!")
print("Your clean file is ready at: /kaggle/working/scene_graphs.json")

/kaggle/working/sce 100%[===================>] 108.74M  2.62MB/s    in 42s     
Download and extraction complete!
Your clean file is ready at: /kaggle/working/scene_graphs.json


In [2]:
!pip install ijson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 3.7 MB/s eta 0:00:00a 0:00:01


In [8]:
import json
import ijson
from tqdm.notebook import tqdm
import gc
from collections import Counter

# ==============================
# Configuration
# ==============================
VG_FILE_PATH = "/kaggle/working/scene_graphs.json"
OUTPUT_PATH = "/kaggle/working/rc_dpo_dataset_balanced.json"

TARGET_PER_PREDICATE = 200   # Enforce balance
ALLOWED_PREDICATES = ["left of", "right of", "above", "below", "on", "under"]

CANONICAL_MAP = {
    "left of": ["left of", "to the left of"],
    "right of": ["right of", "to the right of"],
    "above": ["above"],
    "below": ["below"],
    "on": ["on", "on top of"],
    "under": ["under", "beneath"]
}

SPATIAL_INVERSIONS = {
    "left of": "right of",
    "right of": "left of",
    "above": "below",
    "below": "above",
    "on": "under",
    "under": "on"
}

# ==============================
# Helper Functions
# ==============================
def normalize_predicate(raw_pred):
    if not raw_pred: return None
    raw_pred = raw_pred.lower().strip()
    for canonical, variants in CANONICAL_MAP.items():
        if raw_pred in variants:
            return canonical
    return None

def extract_name(obj):
    if "names" in obj and obj["names"]:
        return obj["names"][0]
    return obj.get("name")

def get_center(obj):
    x, y, w, h = obj.get("x"), obj.get("y"), obj.get("w"), obj.get("h")
    if None in (x, y, w, h):
        return None
    return (x + w / 2, y + h / 2)

def verify_geometry(canonical, sub_center, obj_center):
    sx, sy = sub_center
    ox, oy = obj_center

    # Horizontal
    if canonical == "left of": return sx < ox
    if canonical == "right of": return sx > ox
    
    # Vertical
    if canonical == "above": return sy < oy
    if canonical == "below": return sy > oy
    
    # Vertical with strict horizontal alignment for 'on/under'
    if canonical == "on": return sy < oy and abs(sx - ox) < 150
    if canonical == "under": return sy > oy and abs(sx - ox) < 150

    return False

def is_clean_name(name):
    if not name: return False
    name = name.lower().strip()
    if len(name.split()) > 2 or any(char.isdigit() for char in name) or len(name) < 2:
        return False
    return True

# ==============================
# Extraction Logic
# ==============================
golden_dataset = []
predicate_counter = Counter()
seen_pairs = set()

print("Extracting BALANCED verified spatial pairs using priority search...")

with open(VG_FILE_PATH, "rb") as f:
    objects_stream = ijson.items(f, "item")

    for img_data in tqdm(objects_stream):
        # Stop completely if all quotas are hit
        if all(predicate_counter[p] >= TARGET_PER_PREDICATE for p in ALLOWED_PREDICATES):
            break

        img_id = img_data.get("image_id")
        relationships = img_data.get("relationships", [])
        objects = img_data.get("objects", [])
        object_lookup = {obj["object_id"]: obj for obj in objects}

        # PRIORITY SEARCH: Sort target predicates by which ones we need most
        priority_targets = sorted(ALLOWED_PREDICATES, key=lambda p: predicate_counter[p])
        
        found_for_image = False
        
        # Check for the rarest predicates first in this specific image
        for target_pred in priority_targets:
            if predicate_counter[target_pred] >= TARGET_PER_PREDICATE:
                continue
                
            for rel in relationships:
                canonical = normalize_predicate(rel.get("predicate", ""))
                
                # Only look for the current target predicate to maintain balance
                if canonical != target_pred:
                    continue

                sub_id, obj_id = rel.get("subject_id"), rel.get("object_id")
                subject, obj_node = object_lookup.get(sub_id), object_lookup.get(obj_id)

                if not subject or not obj_node: continue

                sub_name = extract_name(subject)
                obj_name = extract_name(obj_node)

                if not is_clean_name(sub_name) or not is_clean_name(obj_name): continue
                
                sub_name, obj_name = sub_name.lower().strip(), obj_name.lower().strip()
                if sub_name == obj_name: continue

                sub_center, obj_center = get_center(subject), get_center(obj_node)
                if sub_center is None or obj_center is None: continue

                # Geometrical verification
                if not verify_geometry(canonical, sub_center, obj_center): continue

                pair_key = (img_id, sub_name, canonical, obj_name)
                if pair_key in seen_pairs: continue

                # Success - Log and Save
                seen_pairs.add(pair_key)
                predicate_counter[canonical] += 1
                inverted = SPATIAL_INVERSIONS[canonical]

                golden_dataset.append({
                    "image_id": img_id,
                    "predicate": canonical,
                    "chosen": f"The {sub_name} is {canonical} the {obj_name}.",
                    "rejected_spatial": f"The {sub_name} is {inverted} the {obj_name}.",
                    "rejected_role": f"The {obj_name} is {canonical} the {sub_name}.",
                    "rejected_mixed": f"The {obj_name} is {inverted} the {sub_name}."
                })
                
                found_for_image = True
                break 
            
            if found_for_image:
                break

# ==============================
# Save and Finalize
# ==============================
with open(OUTPUT_PATH, "w") as out_f:
    json.dump(golden_dataset, out_f, indent=4)

print(f"\nSaved {len(golden_dataset)} balanced relational pairs.")
print("Final Predicate distribution:", dict(predicate_counter))

gc.collect()

Extracting BALANCED verified spatial pairs using priority search...


0it [00:00, ?it/s]


Saved 1144 balanced relational pairs.
Final Predicate distribution: {'on': 200, 'under': 200, 'below': 200, 'above': 200, 'right of': 144, 'left of': 200}


50

In [ ]:
import json
import random
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO

OUTPUT_PATH = "/kaggle/working/rc_dpo_dataset_balanced.json"

# Helper function to fetch images directly from the official Stanford servers
def fetch_vg_image(image_id):
    # VG splits images across two different URLs. We check both.
    urls = [
        f"https://cs.stanford.edu/people/rak248/VG_100K/{image_id}.jpg",
        f"https://cs.stanford.edu/people/rak248/VG_100K_2/{image_id}.jpg"
    ]
    for url in urls:
        try:
            response = requests.get(url, timeout=5)
            if response.status_code == 200:
                return Image.open(BytesIO(response.content))
        except:
            continue
    return None

def web_manual_check(json_path, n=50):
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    # Pick n random samples from your 3,000 pairs
    samples = random.sample(data, min(n, len(data)))
    
    for i, item in enumerate(samples):
        img_id = item['image_id']
        print(f"Fetching Image {img_id}...")
        
        img = fetch_vg_image(img_id)
        
        if img is None:
            print(f"Skipping {img_id}: Could not fetch from server.")
            continue
            
        # Display the image with the mathematically verified text
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(img)
        
        # Display the "Chosen" text as the title
        plt.title(f"Sample {i+1}/{n} | ID: {img_id}\n\n{item['chosen']}", 
                  fontsize=14, color='darkgreen', fontweight='bold')
        
        # Print the negative constraints below the image
        print(f"\n--- SAMPLE {i+1} (ID: {img_id}) ---")
        print(f"CHOSEN:   {item['chosen']}")
        print(f"SPATIAL:  {item['rejected_spatial']}")
        print(f"ROLE:     {item['rejected_role']}")
        print(f"MIXED:    {item['rejected_mixed']}\n")
        
        plt.axis('off')
        plt.show()

# Run the check on 5 samples
web_manual_check(OUTPUT_PATH, n=50)

In [ ]:
import json
import random
from tqdm import tqdm

# ==============================
# Configuration
# ==============================
INPUT_PATH = "/kaggle/working/rc_dpo_dataset_balanced.json"
OUTPUT_PATH = "/kaggle/working/rc_dpo_dataset_naive.json"

random.seed(42)

COLORS = ["red", "blue", "green", "yellow", "black", "white"]
SIZES = ["small", "large"]
AGES = ["old", "young"]
MATERIALS = ["wooden", "metal"]

ATTRIBUTE_TYPES = ["color", "size", "age", "material"]

# ==============================
# Helper Functions
# ==============================
def get_random_attribute(attr_type, exclude=None):
    pool = {
        "color": COLORS,
        "size": SIZES,
        "age": AGES,
        "material": MATERIALS
    }[attr_type]
    
    candidates = [x for x in pool if x != exclude]
    return random.choice(candidates)

# ==============================
# Dataset A Generation
# ==============================
with open(INPUT_PATH, "r") as f:
    dataset_b = json.load(f)

dataset_a = []

print("Generating Naive Preference Dataset A...")

for sample in tqdm(dataset_b):
    
    chosen_sentence = sample["chosen"]
    
    # Parse subject and object
    # Format: "The sky is above the car."
    parts = chosen_sentence.replace(".", "").split()
    sub_name = parts[1]
    predicate = parts[3]
    obj_name = parts[5]
    
    # Choose attribute type
    attr_type = random.choice(ATTRIBUTE_TYPES)
    
    # Assign attributes to both subject and object
    sub_attr = get_random_attribute(attr_type)
    obj_attr = get_random_attribute(attr_type)
    
    # Build chosen caption with attributes
    chosen_caption = f"The {sub_attr} {sub_name} is {predicate} the {obj_attr} {obj_name}."
    
    # Corrupt ONLY subject attribute
    corrupted_sub_attr = get_random_attribute(attr_type, exclude=sub_attr)
    
    rejected_caption = f"The {corrupted_sub_attr} {sub_name} is {predicate} the {obj_attr} {obj_name}."
    
    dataset_a.append({
        "image_id": sample["image_id"],
        "predicate": sample["predicate"],
        "chosen": chosen_caption,
        "rejected_naive": rejected_caption
    })
print("\nRunning sanity checks...")

# 1️⃣ Size check
assert len(dataset_a) == len(dataset_b), \
    f"Size mismatch: A={len(dataset_a)} B={len(dataset_b)}"

# 2️⃣ Image alignment check
for i in range(len(dataset_a)):
    assert dataset_a[i]["image_id"] == dataset_b[i]["image_id"], \
        f"Image ID mismatch at index {i}"

# 3️⃣ Predicate distribution check
from collections import Counter

pred_a = Counter([x["predicate"] for x in dataset_a])
pred_b = Counter([x["predicate"] for x in dataset_b])

assert pred_a == pred_b, \
    f"Predicate distribution mismatch!\nA:{pred_a}\nB:{pred_b}"

print("✔ Dataset size aligned")
print("✔ Image IDs aligned")
print("✔ Predicate distribution aligned")

# 4️⃣ Ensure chosen != rejected
for i, sample in enumerate(dataset_a):
    assert sample["chosen"] != sample["rejected_naive"], \
        f"Chosen equals rejected at index {i}"

print("✔ All chosen/rejected pairs are different")

# 5️⃣ Ensure structure unchanged (predicate position check)
for i, sample in enumerate(dataset_a):
    chosen_parts = sample["chosen"].replace(".", "").split()
    rejected_parts = sample["rejected_naive"].replace(".", "").split()
    
    # Predicate should be identical
    assert chosen_parts[3] == rejected_parts[3], \
        f"Predicate changed at index {i}"

print("✔ Predicate unchanged in naive corruption")

print("\nAll sanity checks passed.")
# Save
with open(OUTPUT_PATH, "w") as f:
    json.dump(dataset_a, f, indent=4)

print(f"\nSaved {len(dataset_a)} naive preference samples.")

In [ ]:
import json
import random
from collections import defaultdict

INPUT_FILE = "/kaggle/working/rc_dpo_dataset_balanced.json"
OUTPUT_FILE = "/kaggle/working/rc_dpo_dataset_final_augmented.json"

TARGET_PER_PRED = 2000

SPATIAL_INVERSIONS = {
    "left of": "right of",
    "right of": "left of",
    "above": "below",
    "below": "above",
    "on": "under",
    "under": "on"
}

ALL_PREDS = list(SPATIAL_INVERSIONS.keys())

RELATION_TEMPLATES = {
    "left of": [
        "{A} is left of {B}",
        "{A} is to the left of {B}"
    ],
    "right of": [
        "{A} is right of {B}",
        "{A} is to the right of {B}"
    ],
    "above": [
        "{A} is above {B}",
        "{A} is over {B}"
    ],
    "below": [
        "{A} is below {B}",
        "{A} is under {B}"
    ],
    "on": [
        "{A} is on {B}",
        "{A} sits on {B}"
    ],
    "under": [
        "{A} is under {B}",
        "{A} sits under {B}"
    ]
}


def parse_sentence(sentence, pred):

    sentence = sentence.lower().replace(".", "")

    if pred not in sentence:
        return None, None

    try:
        parts = sentence.split(pred)

        subject = parts[0].replace("the ", "").replace(" is ", "").strip()
        obj = parts[1].replace("the ", "").strip()

        return subject, obj

    except:
        return None, None

def generate_sentence(A, B, pred):

    template = random.choice(RELATION_TEMPLATES[pred])
    return "The " + template.format(A=A, B="the " + B) + "."


def create_inverse(example, subject, obj, pred):

    inv = SPATIAL_INVERSIONS[pred]

    return {
        "image_id": example["image_id"],
        "predicate": inv,
        "chosen": generate_sentence(obj, subject, inv),
        "rejected_spatial": generate_sentence(obj, subject, pred),
        "rejected_role": generate_sentence(subject, obj, inv),
        "rejected_mixed": generate_sentence(subject, obj, pred)
    }


def create_role_swap(example, subject, obj, pred):

    inv = SPATIAL_INVERSIONS[pred]

    return {
        "image_id": example["image_id"],
        "predicate": pred,
        "chosen": generate_sentence(obj, subject, pred),
        "rejected_spatial": generate_sentence(obj, subject, inv),
        "rejected_role": generate_sentence(subject, obj, pred),
        "rejected_mixed": generate_sentence(subject, obj, inv)
    }


def create_wrong_predicate(example, subject, obj, pred):

    wrong = random.choice([p for p in ALL_PREDS if p != pred])

    return {
        "image_id": example["image_id"],
        "predicate": pred,
        "chosen": example["chosen"],
        "rejected_spatial": generate_sentence(subject, obj, wrong),
        "rejected_role": generate_sentence(obj, subject, pred),
        "rejected_mixed": generate_sentence(obj, subject, wrong)
    }


counts = defaultdict(int)
final_dataset = []

with open(INPUT_FILE) as f:
    data = json.load(f)

random.shuffle(data)

for ex in data:

   while min(counts.values() or [0]) < TARGET_PER_PRED:

    for ex in data:

        pred = ex["predicate"]

        subject, obj = parse_sentence(ex["chosen"], pred)

        if subject is None:
            continue

        candidates = [
            ex,
            create_inverse(ex, subject, obj, pred),
            create_role_swap(ex, subject, obj, pred),
            create_wrong_predicate(ex, subject, obj, pred)
        ]

        for c in candidates:

            p = c["predicate"]

            if counts[p] < TARGET_PER_PRED:

                final_dataset.append(c)
                counts[p] += 1

        if min(counts.values()) >= TARGET_PER_PRED:
            break

with open(OUTPUT_FILE, "w") as f:
    json.dump(final_dataset, f, indent=4)

print("Final dataset size:", len(final_dataset))
print("Predicate distribution:")

for k,v in counts.items():
    print(k, v)

In [ ]:
import json
import random
import matplotlib.pyplot as plt
from PIL import Image
import requests
from io import BytesIO

OUTPUT_PATH = "/kaggle/working/rc_dpo_dataset_final_augmented.json"

# Helper function to fetch images directly from the official Stanford servers
def fetch_vg_image(image_id):
    # VG splits images across two different URLs. We check both.
    urls = [
        f"https://cs.stanford.edu/people/rak248/VG_100K/{image_id}.jpg",
        f"https://cs.stanford.edu/people/rak248/VG_100K_2/{image_id}.jpg"
    ]
    for url in urls:
        try:
            response = requests.get(url, timeout=5)
            if response.status_code == 200:
                return Image.open(BytesIO(response.content))
        except:
            continue
    return None

def web_manual_check(json_path, n=50):
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    # Pick n random samples from your 3,000 pairs
    samples = random.sample(data, min(n, len(data)))
    
    for i, item in enumerate(samples):
        img_id = item['image_id']
        print(f"Fetching Image {img_id}...")
        
        img = fetch_vg_image(img_id)
        
        if img is None:
            print(f"Skipping {img_id}: Could not fetch from server.")
            continue
            
        # Display the image with the mathematically verified text
        fig, ax = plt.subplots(figsize=(8, 6))
        ax.imshow(img)
        
        # Display the "Chosen" text as the title
        plt.title(f"Sample {i+1}/{n} | ID: {img_id}\n\n{item['chosen']}", 
                  fontsize=14, color='darkgreen', fontweight='bold')
        
        # Print the negative constraints below the image
        print(f"\n--- SAMPLE {i+1} (ID: {img_id}) ---")
        print(f"CHOSEN:   {item['chosen']}")
        print(f"SPATIAL:  {item['rejected_spatial']}")
        print(f"ROLE:     {item['rejected_role']}")
        print(f"MIXED:    {item['rejected_mixed']}\n")
        
        plt.axis('off')
        plt.show()

# Run the check on 5 samples
web_manual_check(OUTPUT_PATH, n=2)

In [16]:
import json
import random
from collections import defaultdict, Counter

# =========================================================
# CONFIG
# =========================================================
INPUT_FILE = "/kaggle/input/datasets/jubaer001/jubaer/rc_dpo_dataset_balanced.json"
OUTPUT_TRAIN = "/kaggle/working/train.json"
OUTPUT_VAL = "/kaggle/working/val.json"
OUTPUT_TEST = "/kaggle/working/test.json"

TARGET_PER_PRED = 600   # will be clipped to feasible maximum automatically
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
SEED = 42

random.seed(SEED)

SPATIAL_INVERSIONS = {
    "left of": "right of",
    "right of": "left of",
    "above": "below",
    "below": "above",
    "on": "under",
    "under": "on"
}

ALL_PREDS = list(SPATIAL_INVERSIONS.keys())

RELATION_TEMPLATES = {
    "left of": [
        "The {A} is left of the {B}.",
        "The {A} is to the left of the {B}."
    ],
    "right of": [
        "The {A} is right of the {B}.",
        "The {A} is to the right of the {B}."
    ],
    "above": [
        "The {A} is above the {B}."
    ],
    "below": [
        "The {A} is below the {B}."
    ],
    "on": [
        "The {A} is on the {B}."
    ],
    "under": [
        "The {A} is under the {B}."
    ]
}

# =========================================================
# HELPERS
# =========================================================
def parse_original_sentence(sentence, pred):
    """
    Robustly parses sentences generated from RELATION_TEMPLATES[pred].
    Supports variants like:
      - The cat is left of the table.
      - The cat is to the left of the table.
    """
    s = sentence.strip().rstrip(".")
    s_lower = s.lower()

    if not s_lower.startswith("the "):
        return None, None

    markers = []
    for template in RELATION_TEMPLATES[pred]:
        # Build relation phrase between {A} and {B}
        relation_part = template.replace("The {A} is ", "").replace("the {B}.", "").replace("{B}.", "").strip()
        marker = f" is {relation_part} the "
        markers.append(marker.lower())

    for marker in markers:
        idx = s_lower.find(marker)
        if idx != -1:
            subject = s[4:idx].strip()
            obj = s[idx + len(marker):].strip()
            if subject and obj:
                return subject, obj

    return None, None

def generate_sentence(subject, obj, pred):
    template = random.choice(RELATION_TEMPLATES[pred])
    return template.format(A=subject, B=obj)

def canonical_key(sample):
    subj_box = tuple(sample["subj_box"]) if sample.get("subj_box") is not None else None
    obj_box = tuple(sample["obj_box"]) if sample.get("obj_box") is not None else None
    return (
        sample["image_id"],
        sample["subject"].lower(),
        sample["object"].lower(),
        sample["predicate"],
        sample["chosen"].lower(),
        subj_box,
        obj_box
    )

def unordered_pair(subject, obj):
    return tuple(sorted([subject.lower(), obj.lower()]))

def make_sample(
    image_id,
    subject,
    object_,
    pred,
    chosen,
    rejected_spatial,
    rejected_role,
    rejected_mixed,
    subj_box=None,
    obj_box=None
):
    return {
        "image_id": image_id,
        "predicate": pred,
        "subject": subject,
        "object": object_,
        "subj_box": subj_box,
        "obj_box": obj_box,
        "chosen": chosen,
        "rejected_spatial": rejected_spatial,
        "rejected_role": rejected_role,
        "rejected_mixed": rejected_mixed
    }

def create_inverse(example, subject, obj, pred, subj_box=None, obj_box=None):
    inv = SPATIAL_INVERSIONS[pred]
    return make_sample(
        image_id=example["image_id"],
        subject=obj,
        object_=subject,
        pred=inv,
        chosen=generate_sentence(obj, subject, inv),
        rejected_spatial=generate_sentence(obj, subject, pred),
        rejected_role=generate_sentence(subject, obj, inv),
        rejected_mixed=generate_sentence(subject, obj, pred),
        subj_box=obj_box,
        obj_box=subj_box
    )

def create_role_swap(example, subject, obj, pred, subj_box=None, obj_box=None):
    inv = SPATIAL_INVERSIONS[pred]
    return make_sample(
        image_id=example["image_id"],
        subject=obj,
        object_=subject,
        pred=pred,
        chosen=generate_sentence(obj, subject, pred),
        rejected_spatial=generate_sentence(obj, subject, inv),
        rejected_role=generate_sentence(subject, obj, pred),
        rejected_mixed=generate_sentence(subject, obj, inv),
        subj_box=obj_box,
        obj_box=subj_box
    )

def create_wrong_predicate(example, subject, obj, pred, subj_box=None, obj_box=None):
    wrong = random.choice([p for p in ALL_PREDS if p != pred])
    return make_sample(
        image_id=example["image_id"],
        subject=subject,
        object_=obj,
        pred=pred,
        chosen=generate_sentence(subject, obj, pred),
        rejected_spatial=generate_sentence(subject, obj, wrong),
        rejected_role=generate_sentence(obj, subject, pred),
        rejected_mixed=generate_sentence(obj, subject, wrong),
        subj_box=subj_box,
        obj_box=obj_box
    )

def create_language_variants(example, subject, obj, pred, subj_box=None, obj_box=None):
    variants = []

    base_rej_sp = generate_sentence(subject, obj, SPATIAL_INVERSIONS[pred])
    base_rej_role = generate_sentence(obj, subject, pred)
    base_rej_mix = generate_sentence(obj, subject, SPATIAL_INVERSIONS[pred])

    for template in RELATION_TEMPLATES[pred]:
        chosen = template.format(A=subject, B=obj)
        variants.append(
            make_sample(
                image_id=example["image_id"],
                subject=subject,
                object_=obj,
                pred=pred,
                chosen=chosen,
                rejected_spatial=base_rej_sp,
                rejected_role=base_rej_role,
                rejected_mixed=base_rej_mix,
                subj_box=subj_box,
                obj_box=obj_box
            )
        )
    return variants

def augment_example(example):
    pred = example["predicate"]
    if pred not in ALL_PREDS:
        return []

    # Prefer explicit subject/object if already present
    subject = example.get("subject")
    obj = example.get("object")

    if subject is None or obj is None:
        subject, obj = parse_original_sentence(example["chosen"], pred)

    if subject is None or obj is None:
        return []

    subj_box = example.get("subj_box")
    obj_box = example.get("obj_box")

    original = make_sample(
        image_id=example["image_id"],
        subject=subject,
        object_=obj,
        pred=pred,
        chosen=example["chosen"],
        rejected_spatial=example["rejected_spatial"],
        rejected_role=example["rejected_role"],
        rejected_mixed=example["rejected_mixed"],
        subj_box=subj_box,
        obj_box=obj_box
    )

    samples = [original]
    samples.append(create_inverse(example, subject, obj, pred, subj_box, obj_box))
    samples.append(create_role_swap(example, subject, obj, pred, subj_box, obj_box))
    samples.append(create_wrong_predicate(example, subject, obj, pred, subj_box, obj_box))
    samples.extend(create_language_variants(example, subject, obj, pred, subj_box, obj_box))

    unique = []
    seen = set()
    for s in samples:
        k = canonical_key(s)
        if k not in seen:
            seen.add(k)
            unique.append(s)

    return unique

# =========================================================
# LOAD INPUT
# =========================================================
with open(INPUT_FILE, "r") as f:
    data = json.load(f)

print("Loaded input examples:", len(data))

# =========================================================
# BUILD CANDIDATE POOL
# =========================================================
candidate_pool = []
for ex in data:
    if ex.get("predicate") not in ALL_PREDS:
        continue
    candidate_pool.extend(augment_example(ex))

# Global dedup
deduped_pool = []
seen = set()
for s in candidate_pool:
    k = canonical_key(s)
    if k not in seen:
        seen.add(k)
        deduped_pool.append(s)

candidate_pool = deduped_pool
random.shuffle(candidate_pool)

print("Candidate pool size:", len(candidate_pool))

candidate_counts = Counter([x["predicate"] for x in candidate_pool])
print("Candidate predicate counts:", dict(candidate_counts))

# Automatically cap target to feasible maximum
feasible_target = min(candidate_counts.values()) if candidate_counts else 0
effective_target = min(TARGET_PER_PRED, feasible_target)

print("Requested TARGET_PER_PRED:", TARGET_PER_PRED)
print("Feasible TARGET_PER_PRED:", feasible_target)
print("Using TARGET_PER_PRED:", effective_target)

# =========================================================
# BALANCED SELECTION
# =========================================================
counts = {p: 0 for p in ALL_PREDS}
balanced_dataset = []
used_keys = set()

for s in candidate_pool:
    p = s["predicate"]
    if counts[p] >= effective_target:
        continue

    k = canonical_key(s)
    if k in used_keys:
        continue

    balanced_dataset.append(s)
    used_keys.add(k)
    counts[p] += 1

    if min(counts.values()) >= effective_target:
        break

print("Balanced dataset size:", len(balanced_dataset))
print("Balanced predicate counts:", counts)

# =========================================================
# SPLIT BY UNORDERED OBJECT PAIR
# =========================================================
pair_to_samples = defaultdict(list)
for s in balanced_dataset:
    pair_to_samples[unordered_pair(s["subject"], s["object"])].append(s)

all_pairs = list(pair_to_samples.keys())
random.shuffle(all_pairs)

n = len(all_pairs)
n_train = int(TRAIN_RATIO * n)
n_val = int(VAL_RATIO * n)

train_pairs = set(all_pairs[:n_train])
val_pairs = set(all_pairs[n_train:n_train + n_val])
test_pairs = set(all_pairs[n_train + n_val:])

train_data, val_data, test_data = [], [], []

for pair, samples in pair_to_samples.items():
    if pair in train_pairs:
        train_data.extend(samples)
    elif pair in val_pairs:
        val_data.extend(samples)
    else:
        test_data.extend(samples)

def show_distribution(name, dataset):
    c = Counter([x["predicate"] for x in dataset])
    print(f"{name} size: {len(dataset)}")
    print(f"{name} predicate distribution: {dict(c)}")

print("\nSplit summary")
show_distribution("Train", train_data)
show_distribution("Val", val_data)
show_distribution("Test", test_data)

# =========================================================
# SAVE
# =========================================================
with open(OUTPUT_TRAIN, "w") as f:
    json.dump(train_data, f, indent=2)

with open(OUTPUT_VAL, "w") as f:
    json.dump(val_data, f, indent=2)

with open(OUTPUT_TEST, "w") as f:
    json.dump(test_data, f, indent=2)

print("\nSaved:")
print(OUTPUT_TRAIN)
print(OUTPUT_VAL)
print(OUTPUT_TEST)

Loaded input examples: 1144
Candidate pool size: 3776
Candidate predicate counts: {'above': 600, 'under': 600, 'right of': 632, 'on': 600, 'below': 600, 'left of': 744}
Requested TARGET_PER_PRED: 600
Feasible TARGET_PER_PRED: 600
Using TARGET_PER_PRED: 600
Balanced dataset size: 3600
Balanced predicate counts: {'left of': 600, 'right of': 600, 'above': 600, 'below': 600, 'on': 600, 'under': 600}

Split summary
Train size: 2501
Train predicate distribution: {'on': 442, 'under': 437, 'right of': 413, 'left of': 387, 'below': 417, 'above': 405}
Val size: 547
Val predicate distribution: {'above': 113, 'below': 112, 'on': 66, 'under': 63, 'left of': 100, 'right of': 93}
Test size: 552
Test predicate distribution: {'under': 100, 'on': 92, 'right of': 94, 'left of': 113, 'above': 82, 'below': 71}

Saved:
/kaggle/working/train.json
/kaggle/working/val.json
/kaggle/working/test.json


In [17]:
num_with_boxes = sum(
    1 for x in balanced_dataset
    if x.get("subj_box") is not None and x.get("obj_box") is not None
)
print("Examples with boxes:", num_with_boxes, "/", len(balanced_dataset))

Examples with boxes: 0 / 3600


In [5]:
import json
import random
import ijson
from collections import defaultdict, Counter

# =========================================================
# CONFIG
# =========================================================
VG_FILE_PATH = "/kaggle/working/scene_graphs.json"   # change if needed
OUTPUT_TRAIN = "/kaggle/working/train.json"
OUTPUT_VAL = "/kaggle/working/val.json"
OUTPUT_TEST = "/kaggle/working/test.json"

TARGET_PER_PRED = 600   # upper cap; actual target will be clipped to feasible count
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
SEED = 42

random.seed(SEED)

# =========================================================
# SPATIAL PREDICATES
# =========================================================
CANONICAL_MAP = {
    "left of": {"left of", "to the left of"},
    "right of": {"right of", "to the right of"},
    "above": {"above"},
    "below": {"below"},
    "on": {"on", "on top of"},
    "under": {"under", "beneath"}
}

SPATIAL_INVERSIONS = {
    "left of": "right of",
    "right of": "left of",
    "above": "below",
    "below": "above",
    "on": "under",
    "under": "on"
}

ALL_PREDS = list(SPATIAL_INVERSIONS.keys())

RELATION_TEMPLATES = {
    "left of": [
        "The {A} is left of the {B}.",
        "The {A} is to the left of the {B}."
    ],
    "right of": [
        "The {A} is right of the {B}.",
        "The {A} is to the right of the {B}."
    ],
    "above": [
        "The {A} is above the {B}."
    ],
    "below": [
        "The {A} is below the {B}."
    ],
    "on": [
        "The {A} is on the {B}."
    ],
    "under": [
        "The {A} is under the {B}."
    ]
}

# =========================================================
# HELPERS
# =========================================================
def normalize_predicate(raw_pred):
    if raw_pred is None:
        return None
    p = raw_pred.strip().lower()
    for canonical, forms in CANONICAL_MAP.items():
        if p in forms:
            return canonical
    return None

def clean_name(name):
    return str(name).strip().lower()

def choose_object_name(obj):
    names = obj.get("names", [])
    if not names:
        return None
    return clean_name(names[0])

def box_from_obj(obj):
    if not all(k in obj for k in ["x", "y", "w", "h"]):
        return None
    return [obj["x"], obj["y"], obj["w"], obj["h"]]

def box_center(box):
    x, y, w, h = box
    return x + w / 2.0, y + h / 2.0

def verify_relation(subj_box, obj_box, predicate):
    """
    Strict but simple geometric verification.
    This filters obvious annotation noise.
    """
    sx, sy = box_center(subj_box)
    ox, oy = box_center(obj_box)

    if predicate == "left of":
        return sx < ox
    if predicate == "right of":
        return sx > ox
    if predicate == "above":
        return sy < oy
    if predicate == "below":
        return sy > oy
    if predicate == "on":
        return sy < oy
    if predicate == "under":
        return sy > oy

    return False

def generate_sentence(subject, obj, pred):
    template = random.choice(RELATION_TEMPLATES[pred])
    return template.format(A=subject, B=obj)

def unordered_pair(subject, obj):
    return tuple(sorted([subject.lower(), obj.lower()]))

def make_sample(
    image_id,
    subject,
    object_,
    pred,
    subj_box,
    obj_box
):
    inv = SPATIAL_INVERSIONS[pred]

    return {
        "image_id": image_id,
        "predicate": pred,
        "subject": subject,
        "object": object_,
        "subj_box": subj_box,
        "obj_box": obj_box,
        "chosen": generate_sentence(subject, object_, pred),
        "rejected_spatial": generate_sentence(subject, object_, inv),
        "rejected_role": generate_sentence(object_, subject, pred),
        "rejected_mixed": generate_sentence(object_, subject, inv)
    }

def canonical_key(sample):
    return (
        sample["image_id"],
        sample["subject"].lower(),
        sample["object"].lower(),
        sample["predicate"],
        tuple(sample["subj_box"]),
        tuple(sample["obj_box"]),
        sample["chosen"].lower()
    )

def swap_sample(sample):
    pred = sample["predicate"]
    inv = SPATIAL_INVERSIONS[pred]
    return {
        "image_id": sample["image_id"],
        "predicate": inv,
        "subject": sample["object"],
        "object": sample["subject"],
        "subj_box": sample["obj_box"],
        "obj_box": sample["subj_box"],
        "chosen": generate_sentence(sample["object"], sample["subject"], inv),
        "rejected_spatial": generate_sentence(sample["object"], sample["subject"], pred),
        "rejected_role": generate_sentence(sample["subject"], sample["object"], inv),
        "rejected_mixed": generate_sentence(sample["subject"], sample["object"], pred)
    }

def create_language_variants(sample):
    pred = sample["predicate"]
    subject = sample["subject"]
    obj = sample["object"]
    subj_box = sample["subj_box"]
    obj_box = sample["obj_box"]
    inv = SPATIAL_INVERSIONS[pred]

    out = []
    base_rej_sp = generate_sentence(subject, obj, inv)
    base_rej_role = generate_sentence(obj, subject, pred)
    base_rej_mix = generate_sentence(obj, subject, inv)

    for template in RELATION_TEMPLATES[pred]:
        out.append({
            "image_id": sample["image_id"],
            "predicate": pred,
            "subject": subject,
            "object": obj,
            "subj_box": subj_box,
            "obj_box": obj_box,
            "chosen": template.format(A=subject, B=obj),
            "rejected_spatial": base_rej_sp,
            "rejected_role": base_rej_role,
            "rejected_mixed": base_rej_mix
        })
    return out

# =========================================================
# EXTRACT FROM RAW VG
# =========================================================
candidate_pool = []

print("Reading raw Visual Genome scene graphs...")

with open(VG_FILE_PATH, "r") as f:
    images = ijson.items(f, "item")

    for img_entry in images:
        image_id = img_entry.get("image_id")
        objects = img_entry.get("objects", [])
        relationships = img_entry.get("relationships", [])

        # Build object lookup: object_id -> {name, box}
        obj_map = {}
        for obj in objects:
            object_id = obj.get("object_id")
            if object_id is None:
                continue

            name = choose_object_name(obj)
            box = box_from_obj(obj)

            if name is None or box is None:
                continue

            obj_map[object_id] = {
                "name": name,
                "box": box
            }

        # Extract valid spatial relationships
        for rel in relationships:
            raw_pred = rel.get("predicate")
            pred = normalize_predicate(raw_pred)
            if pred is None:
                continue

            subj_id = rel.get("subject_id")
            obj_id = rel.get("object_id")

            if subj_id not in obj_map or obj_id not in obj_map:
                continue

            subject = obj_map[subj_id]["name"]
            object_ = obj_map[obj_id]["name"]
            subj_box = obj_map[subj_id]["box"]
            obj_box = obj_map[obj_id]["box"]

            # Skip degenerate self-relations
            if subj_id == obj_id:
                continue

            # Geometric verification
            if not verify_relation(subj_box, obj_box, pred):
                continue

            base = make_sample(
                image_id=image_id,
                subject=subject,
                object_=object_,
                pred=pred,
                subj_box=subj_box,
                obj_box=obj_box
            )

            candidate_pool.append(base)
            candidate_pool.append(swap_sample(base))
            candidate_pool.extend(create_language_variants(base))

print("Raw extracted candidates:", len(candidate_pool))

# =========================================================
# GLOBAL DEDUP
# =========================================================
deduped_pool = []
seen = set()

for s in candidate_pool:
    k = canonical_key(s)
    if k not in seen:
        seen.add(k)
        deduped_pool.append(s)

candidate_pool = deduped_pool
random.shuffle(candidate_pool)

print("Deduped candidate pool size:", len(candidate_pool))

candidate_counts = Counter([x["predicate"] for x in candidate_pool])
print("Candidate predicate counts:", dict(candidate_counts))

if not candidate_counts:
    raise ValueError("No valid candidates found. Check VG path or predicate filtering.")

feasible_target = min(candidate_counts.values())
effective_target = min(TARGET_PER_PRED, feasible_target)

print("Requested TARGET_PER_PRED:", TARGET_PER_PRED)
print("Feasible TARGET_PER_PRED:", feasible_target)
print("Using TARGET_PER_PRED:", effective_target)

# =========================================================
# BALANCED SELECTION
# =========================================================
counts = {p: 0 for p in ALL_PREDS}
balanced_dataset = []

for s in candidate_pool:
    p = s["predicate"]
    if counts[p] >= effective_target:
        continue
    balanced_dataset.append(s)
    counts[p] += 1
    if min(counts.values()) >= effective_target:
        break

print("Balanced dataset size:", len(balanced_dataset))
print("Balanced predicate counts:", counts)

num_with_boxes = sum(
    1 for x in balanced_dataset
    if x.get("subj_box") is not None and x.get("obj_box") is not None
)
print("Examples with boxes:", num_with_boxes, "/", len(balanced_dataset))

# =========================================================
# SPLIT BY UNORDERED OBJECT PAIR
# =========================================================
pair_to_samples = defaultdict(list)
for s in balanced_dataset:
    pair_to_samples[unordered_pair(s["subject"], s["object"])].append(s)

all_pairs = list(pair_to_samples.keys())
random.shuffle(all_pairs)

n = len(all_pairs)
n_train = int(TRAIN_RATIO * n)
n_val = int(VAL_RATIO * n)

train_pairs = set(all_pairs[:n_train])
val_pairs = set(all_pairs[n_train:n_train + n_val])
test_pairs = set(all_pairs[n_train + n_val:])

train_data, val_data, test_data = [], [], []

for pair, samples in pair_to_samples.items():
    if pair in train_pairs:
        train_data.extend(samples)
    elif pair in val_pairs:
        val_data.extend(samples)
    else:
        test_data.extend(samples)

def show_distribution(name, dataset):
    c = Counter([x["predicate"] for x in dataset])
    print(f"{name} size: {len(dataset)}")
    print(f"{name} predicate distribution: {dict(c)}")

print("\nSplit summary")
show_distribution("Train", train_data)
show_distribution("Val", val_data)
show_distribution("Test", test_data)

# =========================================================
# SAVE
# =========================================================
with open(OUTPUT_TRAIN, "w") as f:
    json.dump(train_data, f, indent=2)

with open(OUTPUT_VAL, "w") as f:
    json.dump(val_data, f, indent=2)

with open(OUTPUT_TEST, "w") as f:
    json.dump(test_data, f, indent=2)

print("\nSaved:")
print(OUTPUT_TRAIN)
print(OUTPUT_VAL)
print(OUTPUT_TEST)

Reading raw Visual Genome scene graphs...
Raw extracted candidates: 1404851
Deduped candidate pool size: 800384
Candidate predicate counts: {'on': 384399, 'under': 384399, 'below': 14985, 'above': 14985, 'left of': 877, 'right of': 739}
Requested TARGET_PER_PRED: 600
Feasible TARGET_PER_PRED: 739
Using TARGET_PER_PRED: 600
Balanced dataset size: 3600
Balanced predicate counts: {'left of': 600, 'right of': 600, 'above': 600, 'below': 600, 'on': 600, 'under': 600}
Examples with boxes: 3600 / 3600

Split summary
Train size: 2495
Train predicate distribution: {'on': 429, 'under': 416, 'above': 414, 'below': 420, 'right of': 425, 'left of': 391}
Val size: 570
Val predicate distribution: {'on': 90, 'under': 78, 'below': 94, 'left of': 114, 'right of': 96, 'above': 98}
Test size: 535
Test predicate distribution: {'on': 81, 'under': 106, 'below': 86, 'above': 88, 'right of': 79, 'left of': 95}

Saved:
/kaggle/working/train.json
/kaggle/working/val.json
/kaggle/working/test.json
